## Using propaq

## Basic usage

In order to run a Heisenberg simulation with propaq, you need a qiskit circuit and an observable, in the form of a `SparsePauliOp`. We'll construct a toy example here - 

In [20]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

from qiskit.circuit.library import (
    XXPlusYYGate,
    PhaseGate,
    RZGate,
    CPhaseGate,
    SwapGate,
    XGate
) 
import numpy as np

GATES = [
    (lambda: XXPlusYYGate(
        np.random.uniform(0, 2 * np.pi),
        np.random.uniform(0, 2 * np.pi)
    ), 2),
    (lambda: PhaseGate(np.random.uniform(0, 2 * np.pi)), 1),
    (lambda: RZGate(np.random.uniform(0, 2 * np.pi)), 1),
    (lambda: CPhaseGate(np.random.uniform(0, 2 * np.pi)), 2),
    (lambda: SwapGate(), 2),
    (lambda: XGate(), 1)
]

qc = QuantumCircuit(4)

for _ in range(10): 
    factory, nq = GATES[np.random.randint(len(GATES))]
    gate = factory() 
    qubits = np.random.choice(4, size=nq, replace=False).tolist() 
    qc.append(gate, qubits) 
    
observable = SparsePauliOp.from_list([
    ("XIII", 1.0),
    ("IXII", 1.0),  
    ("IIXI", 1.0),
    ("IIIX", 1.0)
])

Then, we need to convert them into objects recognized by propaq's internals. For this example, we'll implement Majorana propagation.

In [21]:
from propaq.circuits import MajoranaCircuit 
from propaq.datatypes import MajoranaTermSum

mc = MajoranaCircuit.from_qiskit(qc, n_modes = 2 * qc.num_qubits)
mts = MajoranaTermSum.from_sparse_pauli_op(observable)

Now, let's add noise and a truncation strategy, and build the propagator. 

In [22]:
from propaq.noise import UniformNoiseModel, TruncationPolicy

noise = UniformNoiseModel(damping=0.001)
truncation_policy = TruncationPolicy(
    weight_cutoff=10, # Max weight of terms to kee p
    coeff_cutoff=1e-5, # Min coefficient magnitude to keep
)

from propaq.propagators import MajoranaPropagator 

prop = MajoranaPropagator(
    noise = noise,
    truncation = truncation_policy,
    progress_bar=True
)

Now, we can compute the expectation value of the observable by back-propagating it through the circuit and evaluating the resulting term sum.

In [23]:
result = prop.expectation_value(mts, mc, fock_state=0) # default to vacuum state, circuit takes care of state preparation
print("Expectation value:", result.expectation_value)

/tmp/ipykernel_78518/965396031.py:1: UserWarning: propaq: the progress bar term count stays stale between truncation flushes. Reduce `truncation_threshold` for more frequent updates.
  result = prop.expectation_value(mts, mc, fock_state=0) # default to vacuum state, circuit takes care of state preparation
Propagating through gates: 100%|██████████| 21/21 [00:00<00:00, 406.35it/s, terms=4]

Expectation value: 0.0


## Logging 

To gain more information about the propagation process, you can enable logging. Note that this will slow down the simulation. 

In [24]:
from propaq import Logger, LogParser

In [25]:
logger = Logger(filename="propaq.log", log_every=5) # log every 5 gates

In [26]:
prop_log = MajoranaPropagator(
    noise = noise,
    truncation = truncation_policy,
    progress_bar=True,
    logger = logger
)

In [27]:
result = prop_log.expectation_value(mts, mc, fock_state=0) 

/tmp/ipykernel_78518/1348330156.py:1: UserWarning: propaq: the progress bar term count stays stale between truncation flushes. Reduce `truncation_threshold` for more frequent updates.
  result = prop_log.expectation_value(mts, mc, fock_state=0)
Propagating through gates: 100%|██████████| 21/21 [00:00<00:00, 439.10it/s, terms=4]


We should now have a file called `propaq.log` in the current directory, which contains JSON lines of the main propagation events. Each line corresponds to either a gate application or a truncation event, and contains relevant information about the event. This is rather unpleasant to read as-is, so we can use the `LogParser` to extract and visualize the information. 

In [28]:
parser = LogParser("propaq.log")

First, let's look at the gate events. This allows us to see how many terms are being generated in the hashmap and outbox at each step of the propagation.

In [29]:
parser.gate_events

[GateEvent(gate_idx=0, layer_idx=0, map_terms=4, outbox_terms=2),
 GateEvent(gate_idx=5, layer_idx=2, map_terms=4, outbox_terms=21),
 GateEvent(gate_idx=10, layer_idx=4, map_terms=4, outbox_terms=82),
 GateEvent(gate_idx=15, layer_idx=4, map_terms=4, outbox_terms=224),
 GateEvent(gate_idx=20, layer_idx=5, map_terms=4, outbox_terms=1472)]

We can also look at the truncation events, which contain information on the number of terms discarded, the coefficients of the discarded terms, and the truncation thresholds. This can be useful for debugging and tuning the truncation strategy.

In [30]:
parser.truncation_events

[TruncationEvent(gate_idx=21, layer_idx=6, trigger='final', terms_before=48, terms_after=48, terms_discarded=0, discarded_coeff_l1=0.0, discarded_coeff_max=0.0, weight_cutoff=10, coeff_cutoff=1e-05)]

The complete list of available logged information is as follows: 

In [31]:
properties = [
    name
    for name, value in LogParser.__dict__.items()
    if isinstance(value, property)
]
properties

['gate_events',
 'truncation_events',
 'gate_indices',
 'map_terms',
 'outbox_terms',
 'terms_before',
 'terms_after',
 'terms_discarded',
 'discarded_coeff_l1',
 'discarded_coeff_max']